In [4]:
import pandas as pd
import pyodbc
import numpy as np

In [2]:
# 1. Configuración del Servidor

server = 'DESKTOP-O4JIE36\\SQLEXPRESS' 
database = 'RetailSim'

# 2. Cadena de conexión
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'

try:
    # 3. Probamos abrir la puerta
    conexion = pyodbc.connect(conn_str)
    print("¡Conexión exitosa a la base de datos!")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a la base de datos!


### Extraccion

In [3]:
query_extraccion=""" 
    WITH PromedioVenta AS (
    SELECT
        sku,
        AVG(CAST(cantidad_venta AS DECIMAL(10,2))) AS promedio_de_venta
    FROM ventas_diarias
    GROUP BY sku
)
SELECT
    inv.sku,
    inv.cantidad_disponible AS cantidad,
    COALESCE(vta.promedio_de_venta,0) AS promedio_de_venta
FROM inventario_almacen AS inv
LEFT JOIN PromedioVenta AS vta
    ON vta.sku = inv.sku;

"""
df_operativo=pd.read_sql(query_extraccion,conexion)
print(df_operativo.head())

     sku  cantidad  promedio_de_venta
0  10079       166           8.021696
1  10056       195           8.154901
2  10125        10           7.955911
3  10102       284           8.064579
4  10171        62           8.350909


C:\Users\Usuario\AppData\Local\Temp\ipykernel_2704\4081584534.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_operativo=pd.read_sql(query_extraccion,conexion)


### Transformación-Lógica de Negocio

In [11]:
#Calculamos días de inventario
df_operativo['dias_inventario']= np.where(
    df_operativo['promedio_de_venta']>0, 
    df_operativo['cantidad']/df_operativo['promedio_de_venta'],
    999
)

#Redondeamos el resultado de dias a entero
df_operativo['dias_inventario']=np.ceil(df_operativo['dias_inventario'])
df_operativo

,sku,cantidad,promedio_de_venta,dias_inventario,estado_alerta
0,10079,166,8.021696,21.0,Sobre-stock
1,10056,195,8.154901,24.0,Sobre-stock
2,10125,10,7.955911,2.0,Crítico
3,10102,284,8.064579,36.0,Sobre-stock
4,10171,62,8.350909,8.0,Saludable
...,...,...,...,...,...
195,10038,46,7.695296,6.0,Saludable
196,10087,177,7.810756,23.0,Sobre-stock
197,10130,142,7.934693,18.0,Sobre-stock
198,10030,33,8.102844,5.0,Saludable


In [9]:
#Creamos alerta de reposición
# Definimos nuestras reglas de negocio
condiciones=[
    df_operativo['dias_inventario']<=3,
    (df_operativo['dias_inventario']>3) & (df_operativo['dias_inventario']<=15),
    df_operativo['dias_inventario']>15
]
#Definimos las etiquetas para cada condición
etiquetas=['Crítico','Saludable','Sobre-stock']
df_operativo['estado_alerta']=np.select(condiciones,etiquetas,default='Desconocido')
print(df_operativo)


       sku  cantidad  promedio_de_venta  dias_inventario estado_alerta
0    10079       166           8.021696             21.0   Sobre-stock
1    10056       195           8.154901             24.0   Sobre-stock
2    10125        10           7.955911              2.0       Crítico
3    10102       284           8.064579             36.0   Sobre-stock
4    10171        62           8.350909              8.0     Saludable
..     ...       ...                ...              ...           ...
195  10038        46           7.695296              6.0     Saludable
196  10087       177           7.810756             23.0   Sobre-stock
197  10130       142           7.934693             18.0   Sobre-stock
198  10030        33           8.102844              5.0     Saludable
199  10081       109           7.676413             15.0     Saludable

[200 rows x 5 columns]


### Carga de datos

In [12]:
# Guardamos nuestro Data Mart listo para ser consumido
df_operativo.to_csv('inventario_procesado_alertas.csv', index=False, encoding='utf-8')
print("¡Pipeline ETL completado!")

¡Pipeline ETL completado!


In [13]:
# Leer el archivo original
df = pd.read_csv("ventas_diarias.csv")

# Tomar una muestra de 200 filas
muestra = df.head(200)

# Guardarla en la carpeta data
muestra.to_csv(
    "ventas_muestra.csv",
    index=False
)